# Generate Pretend Listening Data
endTime, artistName, trackName, msPlayed

In [7]:
%matplotlib inline

In [8]:
import pandas as pd
import numpy as np

# Random data generator using NumPy
# np.random.seed(42)

# artists = ['Type O Negative', 'Incubus', 'Nine Inch Nails', 'Pantera']
# tracks = {
#     'Type O Negative': ['Stay Out of My Dreams', 'These Three Things', 'World Coming Down', 'Blood & Fire'],
#     'Incubus': ['Stellar', 'Glass', 'Crowded Elevator', 'Nice To Know You', 'Just A Phase'],
#     'Nine Inch Nails': ['Closer', 'In Two', 'Heresy', 'Vessel', 'Reptile'],
#     'Pantera': ['Drag the Waters', '5 Minutes Alone', 'Psycho Holiday', 'Becoming', 'Avoid The Light']
# }
    

# # Pandas dataframe of the dates, using 2024 calendar year
# dates = pd.date_range('2023-01-01', '2024-12-31', freq='h')

# # Empty list of records to be populated
# records = []

# # Loop 300 times to create data
# for i in range(3000):
#     # Select random artist
#     artist = np.random.choice(artists)
#     records.append({
#         'endTime': str(np.random.choice(dates)),
#         'artistName': artist,
#         'trackName': np.random.choice(tracks[artist]),
#         'msPlayed': np.random.randint(30000, 300000)
#     })

# df = pd.DataFrame(records).sort_values('endTime').reset_index(drop=True)
# # Convert the dataframe to a csv file, index=False says do not write the index as a column in the csv
# df.to_json('StreamingHistory.json', orient='records', indent=2)
# # Displays top 5 results
# df.head()

In [9]:
import glob
import pandas as pd

# 1. Match all JSON files in the directory
file_pattern = "StreamingHistory_music_*.json"
json_files = glob.glob(file_pattern)

# 2. Read each file into a list of DataFrames
df_list = [pd.read_json(file) for file in json_files]

# 3. Combine them into a single DataFrame
df = pd.concat(df_list, ignore_index=True)

#Convert the endtime to a datetime value
df['endTime'] = pd.to_datetime(df['endTime'])

#Extract the month
df['month'] = df['endTime'].dt.to_period('M')

df['month_str'] = df['endTime'].dt.strftime('%b %Y')


print(df.dtypes)
df.describe()

# Continue to count listens per month


endTime       datetime64[us]
artistName               str
trackName                str
msPlayed               int64
month              period[M]
month_str                str
dtype: object


,endTime,msPlayed
count,35177,3.517700e+04
mean,2025-12-24 07:04:12.642351,8.267542e+04
min,2025-06-04 21:14:00,0.000000e+00
25%,2025-10-11 19:22:00,9.290000e+02
50%,2026-01-06 16:58:00,1.834000e+03
75%,2026-03-10 16:31:00,1.866670e+05
max,2026-06-05 22:56:00,1.583926e+06
std,NaN,1.338771e+05


In [10]:
# For visualizations
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets #interactive UI elements (dropdowns, sliders, etc)
from IPython.display import display # for rendering

from datetime import datetime

# Drop down to select a month
artist_dropdown = widgets.Dropdown(
    options = ['All'] + sorted(df['artistName'].unique().tolist()),
    description='Artist:',
    value='All'
)

sorted_months = sorted(df['month_str'].unique().tolist(), key=lambda x: datetime.strptime(x, "%b %Y"), reverse=True)

date_dropdown = widgets.Dropdown(
    options = ['All Time'] + sorted_months,
    description = 'Date:',
    value = 'All Time'
)

def update_chart(artist):
    if artist == 'All':
        filtered = df
    else:
        filtered = df[df['artistName'] == artist]

    monthly = filtered.groupby('month')['trackName'].count()

    fig, ax = plt.subplots(figsize=(12,4))
    
    monthly.plot(kind='bar', ax=ax, color='purple', edgecolor='blue')
    ax.set_title(f'Monthly Listens- {artist}')
    ax.set_xlabel('Month')
    ax.set_ylabel('Total Listens')

    plt.tight_layout()
    plt.show()

widgets.interact(update_chart, artist=artist_dropdown)

interactive(children=(Dropdown(description='Artist:', options=('All', '10 Years', '1000 Homo DJs', '16Volt', '…

<function __main__.update_chart(artist)>

In [5]:
def show_summary(artist):
    if artist == 'All':
        filtered = df
    else:
        filtered = df[df['artistName'] == artist]

    print(f"Total Listens:\t {len(filtered)}")
    mostActiveMonth = filtered.groupby('month')['trackName'].count().idxmax()
    print(f"Most Active Month: {mostActiveMonth}")
    mostPlayedTrack = filtered.groupby('trackName')['trackName'].count().idxmax()
    print(f"Most Played Track: {mostPlayedTrack}")

widgets.interact(show_summary, artist=artist_dropdown)

interactive(children=(Dropdown(description='Artist:', options=('All', '10 Years', '1000 Homo DJs', '16Volt', '…

<function __main__.show_summary(artist)>